## Executive Summary

**Purpose:** Check that a helical pair survives its load: pitting of the flanks (contact stress, ISO 6336-2) and fatigue breakage at the root (bending stress, ISO 6336-3) for the required life, plus a static check at the peak torque.

**What it does:** `rate_helical_pair` takes a `HelicalPair`, the operating conditions (`HelicalRatingInputs`) and the materials, and returns the stresses, permissible stresses and safety factors of pinion and gear (`HelicalPairRating`). The calculation is the same Method B route as the planetary rating in `gears.rating`, so the materials and life factors are shared. The helix enters through the base helix in $Z_H$, the overlap in $Z_\varepsilon$ and $Y_\varepsilon$, the helix angle factors $Z_\beta$ and $Y_\beta$, the virtual tooth count in the form factors, and the helical columns of $K_v$ and $K_{H\alpha}$.

**Peak torque:** a short peak is checked as a *static* case. `HelicalRatingInputs(cycles=...)` fixes the number of load cycles, so the life factors take their static values ($Z_{NT} = 1.6$, $Y_{NT} = 2.5$ for case-hardened steel at $\le 10^5$ and $\le 10^3$ cycles).

**Units:** N·m, rpm, N, mm, MPa.

## How It Fits Into the Helical Gear Workflow

**Upstream:** `gears.helical.layout` (`HelicalPair`), `gears.rating` (materials, form factors, rim factor, life factors, elasticity factor).

**Downstream:** `gears.helical.components` (rim thicknesses come back as rating inputs), `gears.helical.design` (the search keeps designs that pass).

In [ ]:
#| default_exp gears.helical.rating

In [ ]:
#| export
from __future__ import annotations

import math
from dataclasses import dataclass

from emachines.gears.rating import (GearMaterial, MATERIALS, GearRating, form_factors, rim_factor,
                                    life_factor_contact, life_factor_bending, _Z_E, _single_pair_factors)
from emachines.gears.helical.layout import HelicalPair

## Notation

| Symbol | Meaning | Units |
|:--:|:--------------------------------|:--:|
| $F_t$ | nominal tangential force at the reference circle, $2000\,T_1/d_1$ | N |
| $u$ | gear ratio $z_2/z_1$ | – |
| $b$ | face width in contact (narrower gear) | mm |
| $K_A, K_v, K_{H\beta}, K_{H\alpha}$ | application, dynamic, face load and transverse load factors | – |
| $K_{F\beta}, K_{F\alpha}$ | the same for bending | – |
| $Z_H$ | zone factor | – |
| $Z_E$ | elasticity factor | $\sqrt{\mathrm{MPa}}$ |
| $Z_\varepsilon$, $Y_\varepsilon$ | contact ratio factors (contact, bending) | – |
| $Z_\beta$, $Y_\beta$ | helix angle factors (contact, bending) | – |
| $Z_B$, $Z_D$ | single pair contact factors of pinion and gear | – |
| $Y_{Fa}, Y_{Sa}$ | form and stress correction factors, load at the tip, on the virtual gear | – |
| $Y_B$ | rim thickness factor | – |
| $Z_{NT}, Y_{NT}$ | life factors | – |
| $\varepsilon_{\alpha n}$ | virtual contact ratio $\varepsilon_\alpha/\cos^2\beta_b$ | – |

## Inputs

`HelicalRatingInputs` holds the pinion torque and speed, the life, and the rating choices. $K_{H\beta}$ defaults to 1.15, for a stiff housing with the gear between two bearings and lightly crowned or ground teeth. A pinion overhung from one bearing, or a long thin shaft, needs 1.3–1.6 or a calculated value (ISO 6336-1 Method C). `gears.helical.components` reports the shaft deflection to help judge it.

In [ ]:
#| export
@dataclass(frozen=True)
class HelicalRatingInputs:
    """
    Operating conditions and rating choices.

    torque_in    : float  Pinion torque [N·m] used for the rating
    n_in         : float  Pinion speed [rpm]
    life_h       : float  Required life [h] at this torque and speed
    K_A          : float  Application factor
    K_Hbeta      : float  Face load factor (see above)
    grade        : int    ISO 1328-1 flank tolerance class
    S_Hmin       : float  Required safety against pitting
    S_Fmin       : float  Required safety against tooth breakage
    flank_finish : str    'ground' or 'hobbed' (sets Z_LvR)
    efficiency   : float  Mesh efficiency, for the gear torque
    cycles       : float | None  Pinion load cycles; overrides life_h (use ≤ 1e3 for a static peak check)
    """
    torque_in: float
    n_in: float
    life_h: float = 10_000.0
    K_A: float = 1.25
    K_Hbeta: float = 1.15
    grade: int = 6
    S_Hmin: float = 1.0
    S_Fmin: float = 1.4
    flank_finish: str = "ground"
    efficiency: float = 0.98
    cycles: float | None = None

## Load Factors

### Dynamic factor $K_v$ (ISO 6336-1 Method C)

$$K_v = 1 + \left(\frac{K_1}{K_A F_t/b} + K_2\right)\frac{z_1 v}{100}\sqrt{\frac{u^2}{1+u^2}},$$

with $v$ the pitch line velocity in m/s and $K_A F_t/b$ no less than 100 N/mm. $K_1$ depends on the flank tolerance class, and $K_2 = 0.0193$ for spur gears and $0.0087$ for helical gears:

| class | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 |
|:--|--:|--:|--:|--:|--:|--:|--:|--:|--:|--:|
| $K_1$ spur | 2.1 | 3.9 | 7.5 | 14.9 | 26.8 | 39.1 | 52.8 | 76.6 | 102.6 | 146.3 |
| $K_1$ helical | 1.9 | 3.5 | 6.7 | 13.3 | 23.9 | 34.8 | 47.0 | 68.2 | 91.4 | 130.3 |

The helical values hold for $\varepsilon_\beta \ge 1$. Below that, $K_v = K_{v\alpha} - \varepsilon_\beta(K_{v\alpha} - K_{v\beta})$ between the spur value $K_{v\alpha}$ and the helical value $K_{v\beta}$.

### Transverse load factor $K_{H\alpha} = K_{F\alpha}$ (ISO 6336-1 Method B, tabulated)

For $K_A F_t/b \ge 100$ N/mm:

| class | ≤ 5 | 6 | 7 | 8 | ≥ 9 |
|:--|:--:|:--:|:--:|:--:|:--:|
| surface hardened, spur | 1.0 | 1.0 | 1.1 | 1.2 | $1/Z_\varepsilon^2 \ge 1.2$ |
| surface hardened, helical | 1.0 | 1.1 | 1.2 | 1.4 | $\varepsilon_{\alpha n} \ge 1.4$ |
| not hardened, spur | 1.0 | 1.0 | 1.0 | 1.1 | $1/Z_\varepsilon^2 \ge 1.2$ |
| not hardened, helical | 1.0 | 1.0 | 1.1 | 1.2 | $\varepsilon_{\alpha n} \ge 1.4$ |

For lightly loaded gears, $K_A F_t/b < 100$ N/mm, the pitch errors are large compared with the tooth deflection. ISO then gives the last column for every class. For a helical pair that is $\varepsilon_{\alpha n} \approx 1.5$–1.7, so a lightly loaded pair can show a *lower* safety factor than at a slightly higher load just above 100 N/mm. That is a property of the tabulated method. The ISO 6336-1 Method B formula, with the pair's real pitch errors, removes the step. In all cases $K_{H\alpha} \le \varepsilon_\gamma/(\varepsilon_\alpha Z_\varepsilon^2)$.

### Face load factor for bending

$$K_{F\beta} = K_{H\beta}^{N_F},\qquad N_F = \frac{(b/h)^2}{1 + b/h + (b/h)^2},$$

with $b/h$ no less than 3.

In [ ]:
#| export
_K1 = {"spur": {3: 2.1, 4: 3.9, 5: 7.5, 6: 14.9, 7: 26.8, 8: 39.1, 9: 52.8, 10: 76.6, 11: 102.6, 12: 146.3},
       "helical": {3: 1.9, 4: 3.5, 5: 6.7, 6: 13.3, 7: 23.9, 8: 34.8, 9: 47.0, 10: 68.2, 11: 91.4, 12: 130.3}}
_K2 = {"spur": 0.0193, "helical": 0.0087}


def dynamic_factor_helical(z1: int, u: float, v: float, wt: float, grade: int, eps_beta: float) -> float:
    "K_v (ISO 6336-1 Method C) for overlap ratio eps_beta. u = z2/z1, v [m/s], wt = K_A·F_t/b [N/mm]."
    gr = int(min(max(grade, 3), 12))
    wt = max(wt, 100.0)
    f = z1 * v / 100 * math.sqrt(u * u / (1 + u * u))
    kv = {k: 1 + (_K1[k][gr] / wt + _K2[k]) * f for k in _K1}
    if eps_beta >= 1:
        return kv["helical"]
    return kv["spur"] - eps_beta * (kv["spur"] - kv["helical"])


def transverse_factor_helical(grade: int, helical: bool, surface_hardened: bool, wt: float,
                              eps_alpha: float, eps_beta: float, beta_b: float, Z_eps: float) -> float:
    "K_Hα = K_Fα (ISO 6336-1 Method B table above), limited to ε_γ / (ε_α Z_ε²)."
    eps_an = eps_alpha / math.cos(beta_b) ** 2
    last = max(eps_an, 1.4) if helical else max(1 / Z_eps ** 2, 1.2)
    if wt < 100 or grade >= 9 + (0 if surface_hardened else 1):
        k = last
    elif helical:
        k = ({6: 1.1, 7: 1.2, 8: 1.4} if surface_hardened else {6: 1.0, 7: 1.1, 8: 1.2, 9: 1.4}).get(grade, 1.0)
    else:
        k = ({6: 1.0, 7: 1.1, 8: 1.2} if surface_hardened else {6: 1.0, 7: 1.0, 8: 1.1, 9: 1.2}).get(grade, 1.0)
    limit = (eps_alpha + eps_beta) / (eps_alpha * Z_eps ** 2)
    return max(1.0, min(k, limit))

In [ ]:
#| hide
from emachines.gears.rating import dynamic_factor
# spur limit reproduces the planetary module's K_v
assert math.isclose(dynamic_factor_helical(17, 3.0, 4.0, 250.0, 6, 0.0), dynamic_factor(17, 3.0, 4.0, 250.0, 6))
assert dynamic_factor_helical(17, 3.0, 4.0, 250.0, 6, 1.2) < dynamic_factor_helical(17, 3.0, 4.0, 250.0, 6, 0.5)
# light load takes the last column; heavy load the class value
assert transverse_factor_helical(6, True, True, 300, 1.5, 1.2, math.radians(14), 1 / math.sqrt(1.5)) == 1.1
_k = transverse_factor_helical(6, True, True, 60, 1.5, 1.2, math.radians(14), 1 / math.sqrt(1.5))
assert math.isclose(_k, 1.5 / math.cos(math.radians(14)) ** 2)

## Contact Stress (ISO 6336-2)

$$\sigma_{H0} = Z_H Z_E Z_\varepsilon Z_\beta\sqrt{\frac{F_t}{d_1 b}\frac{u+1}{u}},\qquad
\sigma_{H1,2} = Z_{B,D}\,\sigma_{H0}\sqrt{K_A K_v K_{H\beta} K_{H\alpha}} .$$

- **Zone factor:** $Z_H = \sqrt{\dfrac{2\cos\beta_b\cos\alpha_{wt}}{\cos^2\alpha_t\sin\alpha_{wt}}}$.
- **Contact ratio factor:** the load spreads over the contact lines of all pairs in mesh.
  $$Z_\varepsilon = \begin{cases}\sqrt{\dfrac{4-\varepsilon_\alpha}{3}(1-\varepsilon_\beta) + \dfrac{\varepsilon_\beta}{\varepsilon_\alpha}} & \varepsilon_\beta < 1\\[2ex] \sqrt{1/\varepsilon_\alpha} & \varepsilon_\beta \ge 1\end{cases}$$
- **Helix angle factor:** $Z_\beta = 1/\sqrt{\cos\beta}$ (ISO 6336-2:2019). The 2006 edition used $\sqrt{\cos\beta}$. The 2019 value is about 3.5 % higher at 15°, so it is the conservative choice.
- **Single pair factors:** $Z_B$ and $Z_D$ move the stress from the pitch point to the inner point of single contact of pinion and gear. They come from the spur values $M_1$, $M_2$ of the transverse equivalents, and fade out with overlap: $Z_B = M_1 - \varepsilon_\beta(M_1 - 1)$, with $Z_B = 1$ once $\varepsilon_\beta \ge 1$ (and the same for $Z_D$).

The permissible stress is $\sigma_{HP} = \sigma_{H\lim} Z_{NT} Z_{LvR}/S_{H\min}$, with $Z_{LvR} = 0.92$ for ground and $0.85$ for hobbed flanks, as in `gears.rating`. The safety factor is $S_H = \sigma_{H\lim}Z_{NT}Z_{LvR}/\sigma_H$.

## Bending Stress (ISO 6336-3, Method B)

$$\sigma_F = \frac{F_t}{b\,m_n}\,Y_{Fa}Y_{Sa}\,Y_\varepsilon Y_\beta Y_B\;K_A K_v K_{F\beta}K_{F\alpha} .$$

- **Form factors** $Y_{Fa}$ and $Y_{Sa}$ (load at the tip) come from `gears.rating.form_factors` applied to the **virtual gear** ($z_n$ teeth, normal module and pressure angle). That makes the 30° tangent section match the normal section of the helical tooth.
- **Contact ratio factor:** $Y_\varepsilon = 0.25 + 0.75/\varepsilon_{\alpha n}$, with $\varepsilon_{\alpha n} = \varepsilon_\alpha/\cos^2\beta_b$.
- **Helix angle factor:** the oblique contact line loads the root less than a line along the face does.
  $$Y_\beta = 1 - \varepsilon_\beta\frac{\beta}{120°} \ge \max(1 - 0.25\varepsilon_\beta,\ 0.75),\qquad \varepsilon_\beta \to \min(\varepsilon_\beta, 1),\ \beta \to \min(\beta, 30°) .$$
- **Face width:** each gear uses its own face width, but no more than the mate's width plus one module on each side.

The permissible stress is $\sigma_{FP} = \sigma_{F\lim}Y_{ST}Y_{NT}/S_{F\min}$, with $Y_{ST} = 2$ and the gear loaded in one direction ($Y_M = 1$). A drive that reverses under full load, such as a robot joint, sees alternating bending. Use about 0.7 of the bending margin for it, or rate with $Y_M = 0.7$ (`alternating=True`).

**Life:** the pinion sees $N_1 = 60\,L_h\,n_1$ load cycles and the gear $N_1/u$.

In [ ]:
#| export
@dataclass(frozen=True)
class HelicalPairRating:
    """
    Result of `rate_helical_pair`.

    T1, T2    : pinion and gear torques [N·m]
    F_t       : tangential force at the reference circle [N] (ISO 6336 nominal load)
    v         : pitch line velocity [m/s]
    factors   : the load and geometry factors used
    gears     : GearRating of pinion and gear
    """
    T1: float
    T2: float
    F_t: float
    v: float
    factors: dict
    gears: tuple[GearRating, ...]
    S_Hmin: float
    S_Fmin: float

    @property
    def S_H(self) -> float: return min(r.S_H for r in self.gears)

    @property
    def S_F(self) -> float: return min(r.S_F for r in self.gears)

    @property
    def ok(self) -> bool: return self.S_H >= self.S_Hmin and self.S_F >= self.S_Fmin

    @property
    def limiting(self) -> str:
        "Which gear and failure mode has the lowest margin over its required safety."
        rows = [(r.S_H / self.S_Hmin, f"pitting of the {r.gear}") for r in self.gears]
        rows += [(r.S_F / self.S_Fmin, f"root breakage of the {r.gear}") for r in self.gears]
        return min(rows)[1]

    def table(self) -> str:
        "Fixed-width text table of the results."
        hdr = (f"{'gear':8s} {'cycles':>9s} {'σH':>7s} {'σHP':>7s} {'S_H':>6s} {'σF':>7s} {'σFP':>7s} {'S_F':>6s} "
               f"{'Y_Fa':>5s} {'Y_Sa':>5s} {'Y_B':>5s} {'Z_NT':>5s} {'Y_NT':>5s}")
        lines = [hdr]
        for r in self.gears:
            lines.append(f"{r.gear:8s} {r.cycles:9.2e} {r.sigma_H:7.0f} {r.sigma_HP:7.0f} {r.S_H:6.2f} "
                         f"{r.sigma_F:7.0f} {r.sigma_FP:7.0f} {r.S_F:6.2f} {r.Y_Fa:5.2f} {r.Y_Sa:5.2f} {r.Y_B:5.2f} "
                         f"{r.Z_NT:5.2f} {r.Y_NT:5.2f}")
        lines.append(f"min S_H = {self.S_H:.2f} (need {self.S_Hmin}), min S_F = {self.S_F:.2f} (need {self.S_Fmin}); "
                     f"limited by {self.limiting}")
        return "\n".join(lines)


def rate_helical_pair(pair: HelicalPair, inp: HelicalRatingInputs,
                      materials: dict[str, GearMaterial] | None = None,
                      rims: dict[str, float | None] | None = None, alternating: bool = False) -> HelicalPairRating:
    """
    ISO 6336 rating of a helical pair (see above).

    materials   : {'pinion', 'gear'} → GearMaterial (default: both case carburized)
    rims        : {'pinion', 'gear'} → rim thickness under the tooth roots [mm] (None or missing: solid)
    alternating : the teeth are loaded on both flanks (reversing drive): Y_M = 0.7
    """
    mats = {"pinion": MATERIALS["case_carburized"], "gear": MATERIALS["case_carburized"]}
    mats.update(materials or {})
    rims = rims or {}
    p, g, mc = pair.pinion, pair.gear, pair.mesh
    u = pair.ratio
    T1 = inp.torque_in
    F_t = 2000 * T1 / p.d
    v = math.pi * p.d * abs(inp.n_in) / 60_000
    b = min(p.b, g.b)
    ea, eb = mc.eps_alpha, mc.eps_beta
    helical = p.beta > 0
    wt = inp.K_A * F_t / b
    Kv = dynamic_factor_helical(p.z, u, v, wt, inp.grade, eb)
    if eb < 1:
        Ze = math.sqrt((4 - ea) / 3 * (1 - eb) + eb / ea)
    else:
        Ze = math.sqrt(1 / ea)
    surf = all(m.kind != "through" for m in mats.values())
    KHa = transverse_factor_helical(inp.grade, helical, surf, wt, ea, eb, p.beta_b, Ze)
    hd = (p.ha + p.hf) * p.mn
    bh = max(b / hd, 3.0)
    NF = bh ** 2 / (1 + bh + bh ** 2)
    KFb = inp.K_Hbeta ** NF
    awt = mc.alpha_wt
    ZH = math.sqrt(2 * math.cos(p.beta_b) * math.cos(awt) / (math.cos(p.alpha_t) ** 2 * math.sin(awt)))
    ZE = _Z_E(mats["pinion"], mats["gear"])
    Zb = 1 / math.sqrt(math.cos(p.beta))
    sH0 = ZH * ZE * Ze * Zb * math.sqrt(F_t / (p.d * b) * (u + 1) / u)
    M1, M2 = _single_pair_factors(p.transverse(), g.transverse(), awt, ea)
    ZB, ZD = (1.0, 1.0) if eb >= 1 else (max(1.0, M1 - eb * (M1 - 1)), max(1.0, M2 - eb * (M2 - 1)))
    KH = inp.K_A * Kv * inp.K_Hbeta * KHa
    KF = inp.K_A * Kv * KFb * KHa
    ean = ea / math.cos(p.beta_b) ** 2
    Ye = 0.25 + 0.75 / ean
    eb1, beta_deg = min(eb, 1.0), min(math.degrees(p.beta), 30.0)
    Yb = max(1 - eb1 * beta_deg / 120, 1 - 0.25 * eb1, 0.75)
    N1 = inp.cycles if inp.cycles is not None else 60 * inp.life_h * abs(inp.n_in)
    cycles = {"pinion": N1, "gear": N1 / u}
    zlvr = {"ground": 0.92, "hobbed": 0.85}[inp.flank_finish]
    YM = 0.7 if alternating else 1.0
    out = []
    for key, gg, Zbd, mate in (("pinion", p, ZB, g), ("gear", g, ZD, p)):
        mat = mats[key]
        sH = Zbd * sH0 * math.sqrt(KH)
        ZNT = life_factor_contact(cycles[key], mat.kind)
        sHlim = mat.sigma_Hlim * ZNT * zlvr
        vg = gg.virtual()
        ff = form_factors(vg)
        YB = rim_factor(vg, rims.get(key))
        b_F = min(gg.b, mate.b + 2 * gg.mn)
        sF = F_t / (b_F * gg.mn) * ff["Y_Fa"] * ff["Y_Sa"] * Ye * Yb * YB * KF
        YNT = life_factor_bending(cycles[key], mat.kind)
        sFlim = mat.sigma_Flim * 2.0 * YNT * YM
        out.append(GearRating(key, "pinion–gear", cycles[key], sH, sHlim / inp.S_Hmin, sHlim / sH,
                              sF, sFlim / inp.S_Fmin, sFlim / sF, ff["Y_Fa"], ff["Y_Sa"], YB, ZNT, YNT))
    factors = {"K_A": inp.K_A, "K_v": Kv, "K_Hbeta": inp.K_Hbeta, "K_Fbeta": KFb, "K_Halpha": KHa,
               "Z_H": ZH, "Z_E": ZE, "Z_eps": Ze, "Z_beta": Zb, "Z_B": ZB, "Z_D": ZD, "Y_eps": Ye, "Y_beta": Yb,
               "Y_M": YM, "Z_LvR": zlvr}
    return HelicalPairRating(T1, T1 * u * inp.efficiency, F_t, v, factors, tuple(out), inp.S_Hmin, inp.S_Fmin)

In [ ]:
import math
from emachines.gears.helical.layout import HelicalPair

pair = HelicalPair.build(17, 52, mn=1.5, beta=math.radians(15), b=20)
r = rate_helical_pair(pair, HelicalRatingInputs(torque_in=20.0, n_in=3000.0, life_h=2000))
print(r.table())
print({k: round(v, 3) for k, v in r.factors.items()})

In [ ]:
#| hide
# stresses scale as √T (contact) and T (bending) when no factor changes
_r2 = rate_helical_pair(pair, HelicalRatingInputs(torque_in=40.0, n_in=3000.0, life_h=2000))
assert all(math.isclose(a.sigma_F / b_.sigma_F, 2 * _r2.factors["K_v"] / r.factors["K_v"] * _r2.factors["K_Halpha"] / r.factors["K_Halpha"], rel_tol=1e-9)
           for a, b_ in zip(_r2.gears, r.gears))
# helix factors
assert math.isclose(r.factors["Z_beta"], 1 / math.sqrt(math.cos(math.radians(15))))
assert pair.mesh.eps_beta >= 1 and r.factors["Z_B"] == 1.0 and math.isclose(r.factors["Z_eps"], 1 / math.sqrt(pair.mesh.eps_alpha))
assert math.isclose(r.factors["Y_beta"], 1 - 15 / 120)
# a static peak case uses the static life factors
_rs = rate_helical_pair(pair, HelicalRatingInputs(torque_in=40.0, n_in=3000.0, K_A=1.0, cycles=1e3))
assert _rs.gears[0].Z_NT == 1.6 and _rs.gears[0].Y_NT == 2.5
# reversing drive lowers the bending margin by Y_M
_ra = rate_helical_pair(pair, HelicalRatingInputs(torque_in=20.0, n_in=3000.0, life_h=2000), alternating=True)
assert math.isclose(_ra.S_F, 0.7 * r.S_F)
# β → 0: spur equivalence of the contact stress (no overlap, Z_beta = 1)
_sp = HelicalPair.build(17, 52, mn=1.5, beta=0.0, b=20)
_rsp = rate_helical_pair(_sp, HelicalRatingInputs(torque_in=20.0, n_in=3000.0, life_h=2000))
assert _rsp.factors["Z_beta"] == 1.0 and _rsp.factors["Y_beta"] == 1.0
# above 100 N/mm the helical pair at the same size is stronger than the spur pair
_h = rate_helical_pair(pair, HelicalRatingInputs(torque_in=40.0, n_in=3000.0, life_h=2000))
_s = rate_helical_pair(_sp, HelicalRatingInputs(torque_in=40.0, n_in=3000.0, life_h=2000))
assert _h.S_H > _s.S_H and _h.S_F > _s.S_F, (_h.S_H, _s.S_H, _h.S_F, _s.S_F)

### Helix angle and strength

The figure below rates the same 17/52 pair ($m_n = 1.5$, $b = 20$ mm, $x_1 = 0.3$, $x_2 = 0$) at 40 N·m over the helix angle. Below $\varepsilon_\beta = 1$ (about 13° here) the factors pull both ways. $Z_\varepsilon$ and $Y_\beta$ improve with overlap, while $K_{H\alpha}$ steps up to its helical value and $\varepsilon_\alpha$ falls. Past that point, contact safety gains a little and bending safety keeps rising with $Y_\beta$ up to 30°. The axial force that the bearings must take rises steadily with $\tan\beta$. That is why the helix angle is a judgement rather than an optimum.

In [ ]:
#| echo: false
import numpy as np
import matplotlib.pyplot as plt

def _beta_figure():
    betas = np.arange(0, 31, 1.0)
    SH, SF, FA = [], [], []
    for bd in betas:
        pr = HelicalPair.build(17, 52, mn=1.5, beta=math.radians(bd), b=20, x1=0.3, x2=0.0)
        rr = rate_helical_pair(pr, HelicalRatingInputs(torque_in=40.0, n_in=3000.0, life_h=2000))
        SH.append(rr.S_H); SF.append(rr.S_F); FA.append(pr.forces(40.0)["F_a"] / pr.forces(40.0)["F_t"])
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(betas, SH, label="S_H (pitting)", color="#1f5fa8")
    ax.plot(betas, SF, label="S_F (bending)", color="#b8322a")
    ax.set_xlabel("helix angle β [°]"); ax.set_ylabel("safety factor"); ax.grid(alpha=0.3)
    ax2 = ax.twinx(); ax2.plot(betas, FA, color="#2a8a4a", ls="--", label="F_a / F_t"); ax2.set_ylabel("F_a / F_t")
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, frameon=False, loc="upper left")
    fig.tight_layout(); plt.show()
_beta_figure()

## References

1. ISO 6336-1:2006, *Calculation of load capacity of spur and helical gears — Part 1: Basic principles, introduction and general influence factors* ($K_v$ Method C, $K_{H\alpha}$, $K_{F\beta}$).
2. ISO 6336-2:2019, *Part 2: Calculation of surface durability (pitting)* ($Z_H$, $Z_\varepsilon$, $Z_\beta$, $Z_B$, $Z_D$).
3. ISO 6336-3:2006, *Part 3: Calculation of tooth bending strength* ($Y_{Fa}$, $Y_{Sa}$ on the virtual gear, $Y_\varepsilon$, $Y_\beta$, $Y_B$).
4. ISO 6336-5:2016, *Part 5: Strength and quality of materials* (allowable stresses, via `gears.rating.MATERIALS`).
5. G. Niemann, H. Winter, *Maschinenelemente*, Band II, Springer, 2nd ed., 1989.